In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
import gc

# 1. LOAD DATA
inter_url = 'https://raw.githubusercontent.com/richiceri/IWC-kaggle/refs/heads/main/interactions_train.csv'
items_url = 'https://raw.githubusercontent.com/richiceri/IWC-kaggle/refs/heads/main/items.csv'

print("📥 Loading data from GitHub...")
interactions = pd.read_csv(inter_url)
items = pd.read_csv(items_url)

n_users = interactions.u.max() + 1
n_items = items.i.max() + 1

# Prep content-based data (Global item features)
print("🧠 Building metadata vectors...")
items_s = items.sort_values('i').set_index('i').reindex(range(n_items))
items_s['meta'] = (items_s['Author'].fillna('') + " ") * 2 + items_s['Subjects'].fillna('')
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_m = tfidf.fit_transform(items_s['meta'])
c_sim = cosine_similarity(tfidf_m).astype(np.float32)

# 2. CROSS-VALIDATION LOOP
kf = KFold(n_splits=5, shuffle=True, random_state=42)
metrics = {
    'user_user': {'precision': [], 'recall': []},
    'item_item': {'precision': [], 'recall': []},
    'hybrid': {'precision': [], 'recall': []}
}

print("\n🚀 Starting 5-Fold Cross-Validation Validation...")
for fold, (train_idx, test_idx) in enumerate(kf.split(interactions)):
    print(f"\n--- Processing Fold {fold + 1}/5 ---")
    train_inter = interactions.iloc[train_idx].copy()

    # Group test items by user for ground-truth calculation
    test_inter = interactions.iloc[test_idx]
    ground_truth = test_inter.groupby('u')['i'].apply(set).to_dict()

    # Compute temporal weights for the training subset
    max_t, min_t = train_inter.t.max(), train_inter.t.min()
    train_inter['time_weight'] = 0.5 + 0.5 * (train_inter.t - min_t) / (max_t - min_t + 1e-9)

    # Construct interaction matrix
    matrix = np.zeros((n_users, n_items), dtype=np.float32)
    matrix[train_inter.u.values, train_inter.i.values] = train_inter.time_weight.values

    # Brain 1: Pure User-User CF
    u_sim = cosine_similarity(matrix).astype(np.float32)
    u_pred = u_sim.dot(matrix) / (np.abs(u_sim).sum(axis=1)[:, np.newaxis] + 1e-9)
    del u_sim; gc.collect()

    # Brain 2: Pure Item-Item CF
    i_sim = cosine_similarity(matrix.T).astype(np.float32)
    i_pred = matrix.dot(i_sim) / (np.abs(i_sim).sum(axis=0) + 1e-9)

    # Brain 3: Pure Content-Based Predictor
    c_pred = matrix.dot(c_sim) / (np.abs(c_sim).sum(axis=0) + 1e-9)

    # Combine Into Adaptive Segmented Hybrid (Your 1689 Pipeline)
    user_counts = train_inter.groupby('u')['i'].count()
    new_users = user_counts[user_counts <= 5].index
    power_users = user_counts[user_counts > 5].index

    hybrid_pred = np.zeros_like(u_pred)
    hybrid_pred[new_users] = (0.20 * u_pred[new_users]) + (0.30 * i_pred[new_users]) + (0.50 * c_pred[new_users])
    hybrid_pred[power_users] = (0.45 * u_pred[power_users]) + (0.40 * i_pred[power_users]) + (0.15 * c_pred[power_users])

    # Last Book Nudge
    last_books = train_inter.sort_values(['u', 't']).groupby('u').tail(1).set_index('u')['i']
    for u_id, last_i in last_books.items():
        hybrid_pred[u_id] *= (1 + 0.1 * i_sim[last_i])
    del i_sim; gc.collect()

    # History Repeat Interactions Boost & Popularity Scaling
    hybrid_pred[matrix > 0] *= 2.0
    pop = train_inter['i'].value_counts().reindex(range(n_items), fill_value=0).values
    pop_boost = np.log1p(pop) / (np.log1p(pop).max() + 1e-9)
    hybrid_pred *= (1 + 0.15 * pop_boost)

    # EVALUATE FOLD METRICS
    for model_name, preds in [('user_user', u_pred), ('item_item', i_pred), ('hybrid', hybrid_pred)]:
        p_list, r_list = [], []
        for u_id, actual_set in ground_truth.items():
            if not actual_set: continue
            top_10 = np.argsort(preds[u_id, :])[-10:][::-1]

            hits = len(actual_set.intersection(top_10))
            p_list.append(hits / 10.0)
            r_list.append(hits / len(actual_set))

        metrics[model_name]['precision'].append(np.mean(p_list))
        metrics[model_name]['recall'].append(np.mean(r_list))

# 3. PRINT DISCOVERED METRIC ARRAYS
print("\n📊 FINAL CROSS-VALIDATION MATRIX VALUES (Ready for README):")
print("-" * 75)
for m in ['user_user', 'item_item', 'hybrid']:
    print(f"{m.upper()} -> Avg Precision@10: {np.mean(metrics[m]['precision']):.4f} | Avg Recall@10: {np.mean(metrics[m]['recall']):.4f}")

📥 Loading data from GitHub...
🧠 Building metadata vectors...

🚀 Starting 5-Fold Cross-Validation Validation...

--- Processing Fold 1/5 ---

--- Processing Fold 2/5 ---

--- Processing Fold 3/5 ---

--- Processing Fold 4/5 ---

--- Processing Fold 5/5 ---

📊 FINAL CROSS-VALIDATION MATRIX VALUES (Ready for README):
---------------------------------------------------------------------------
USER_USER -> Avg Precision@10: 0.0664 | Avg Recall@10: 0.3079
ITEM_ITEM -> Avg Precision@10: 0.0638 | Avg Recall@10: 0.2778
HYBRID -> Avg Precision@10: 0.0708 | Avg Recall@10: 0.3137


In [2]:
# ============================================================
# 4. FINAL PRODUCTION RUN (100% DATA TRAINING)
# ============================================================
print("\n🏗️ Starting Final Production Run on Full Dataset...")

# 1. Re-compute Global Temporal Weights across all available history
max_t = interactions.t.max()
min_t = interactions.t.min()
interactions['time_weight'] = 0.5 + 0.5 * (interactions.t - min_t) / (max_t - min_t + 1e-9)

# 2. Re-build Master Interaction Matrix
full_matrix = np.zeros((n_users, n_items), dtype=np.float32)
full_matrix[interactions.u.values, interactions.i.values] = interactions.time_weight.values

# 3. Re-train the Three Brains on complete data footprint
print("🧠 Re-calculating full User-User connections...")
u_sim_full = cosine_similarity(full_matrix).astype(np.float32)
u_pred_full = u_sim_full.dot(full_matrix) / (np.abs(u_sim_full).sum(axis=1)[:, np.newaxis] + 1e-9)
del u_sim_full; gc.collect()

print("🧠 Re-calculating full Item-Item space...")
i_sim_full = cosine_similarity(full_matrix.T).astype(np.float32)
i_pred_full = full_matrix.dot(i_sim_full) / (np.abs(i_sim_full).sum(axis=0) + 1e-9)

print("🧠 Re-calculating full Metadata Content mapping...")
c_pred_full = full_matrix.dot(c_sim) / (np.abs(c_sim).sum(axis=0) + 1e-9)
gc.collect()

# 4. Apply Dynamic Segmented Weighting
print("⚖️ Applying adaptive weights across profile horizons...")
user_counts_full = interactions.groupby('u')['i'].count()
new_users_full = user_counts_full[user_counts_full <= 5].index
power_users_full = user_counts_full[user_counts_full > 5].index

final_preds_full = np.zeros_like(u_pred_full)
final_preds_full[new_users_full] = (0.20 * u_pred_full[new_users_full]) + (0.30 * i_pred_full[new_users_full]) + (0.50 * c_pred_full[new_users_full])
final_preds_full[power_users_full] = (0.45 * u_pred_full[power_users_full]) + (0.40 * i_pred_full[power_users_full]) + (0.15 * c_pred_full[power_users_full])

# 5. Inject Last-Book Contextual Nudges
print("🎯 Executing next-in-series context nudges...")
last_books_full = interactions.sort_values(['u', 't']).groupby('u').tail(1).set_index('u')['i']
for u_id, last_i in last_books_full.items():
    final_preds_full[u_id] *= (1 + 0.1 * i_sim_full[last_i])
del i_sim_full; gc.collect()

# 6. Apply Historic Re-read Multipliers & Global Popularity Scaling
print("📈 Scaling by popularity metrics and repeat checkouts...")
final_preds_full[full_matrix > 0] *= 2.0  # The 54% historical re-read booster

pop_full = interactions['i'].value_counts().reindex(range(n_items), fill_value=0).values
pop_boost_full = np.log1p(pop_full) / (np.log1p(pop_full).max() + 1e-9)
final_preds_full *= (1 + 0.15 * pop_boost_full)

# 7. EXPORT TOP 10 RANKINGS TO CSV
print("💾 Generating final submission file...")
submission_rows = []
for u_id in range(n_users):
    top_10 = np.argsort(final_preds_full[u_id, :])[-10:][::-1]
    submission_rows.append([u_id, " ".join(map(str, top_10))])

submission_df = pd.DataFrame(submission_rows, columns=['user_id', 'recommendation'])
submission_df.to_csv('submission_adaptive_hybrid.csv', index=False)

print("🏆 ✅ submission_adaptive_hybrid.csv successfully created!")
print("This file contains the complete top-10 optimized predictions matching your 1689 leaderboard benchmark.")


🏗️ Starting Final Production Run on Full Dataset...
🧠 Re-calculating full User-User connections...
🧠 Re-calculating full Item-Item space...
🧠 Re-calculating full Metadata Content mapping...
⚖️ Applying adaptive weights across profile horizons...
🎯 Executing next-in-series context nudges...
📈 Scaling by popularity metrics and repeat checkouts...
💾 Generating final submission file...
🏆 ✅ submission_adaptive_hybrid.csv successfully created!
This file contains the complete top-10 optimized predictions matching your 1689 leaderboard benchmark.


In [ ]:
from google.colab import files
files.download('submission_adaptive_hybrid.csv')